# Expert Iteration — V9 (v9_dataBSG_250k)

Aplica Expert Iteration (ExIt) al modelo V9 pre-entrenado con datos BSG.

**Idea:** En cada iteración, el solver **BSG (Beam Search Greedy, beam=5)** actúa como oráculo experto — etiqueta los movimientos óptimos en un nuevo lote de instancias generadas con seed distinto. El dataset crece monotónicamente combinando el dataset SL original (BSG_250k) + los lotes ExIt-BSG acumulados. Se hace fine-tune del modelo sobre este replay buffer en cada iteración.

**Pipeline:**
1. Cargar modelo base y dataset BSG_250k (replay buffer permanente)
2. Evaluar baseline en CVS
3. Loop por iteración:
   - Generar instancias frescas (seed = 42 + iter)
   - Etiquetar con BSG (beam=5)
   - Construir dataset combinado (BSG base + ExIt-BSG acumulados)
   - Fine-tune
   - Evaluar en CVS + guardar checkpoint

In [ ]:
import sys, os
SRC_PATH = os.path.normpath(os.path.join(os.path.abspath(''), '..', 'src'))
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

CONFIG = {
    "model_name"       : "v9_dataBSG_250k",
    "output_name"      : "v9_ExIt",
    "iterations"       : 5,
    "max_steps"        : 100,
    "epochs_per_iter"  : 15,
    "batch_size"       : 64,
    "learning_rate"    : 1e-4,             # LR del SL original — fine-tune sobre dataset mezclado
    "weight_decay"     : 1e-4,
    "patience"         : 5,                # con LR mayor, plateau es más rápido
    "test_ratio"       : 0.1,              # split por proporción — usa todo el dataset
    "instances_per_set": 5000,
    "bsg_beams"        : 5,                # BSG como expert real (FRG con beam=5)
    "bsg_base_datasets": [                 # replay buffer: dataset SL original mezclado por iteración
        "E4-15-H5_V9_BSG",
        "E5-25-H7_V9_BSG",
        "E6-45-H10_V9_BSG",
        "E7-30-H6_V9_BSG",
    ],
    "instance_sets": [
        ("ExIt_E4",   4,  5, 15),
        ("ExIt_E5",   5,  7, 25),
        ("ExIt_E6",   6, 10, 45),
        ("ExIt_E7",   7,  6, 30),
        ("ExIt_5x5",  5,  5, 20),
        ("ExIt_6x6",  6,  6, 30),
        ("ExIt_7x7",  7,  7, 42),
        ("ExIt_10x8", 10, 8, 72),
        ("ExIt_10x7", 10, 7, 60),
        ("ExIt_8x7",  8,  7, 48),
    ],
}
print("Configuración cargada.")

## 1. Cargar modelo base

In [ ]:
from models.cpmp_transformer_v9 import CPMPTransformer
from training.training import load_model, train, save_model
from generation.adapters import EnrichedStackMatrix5DAdapter, DefaultMovesAdapter
from generation.data import generate_data
from generation.instances import generate_instances
from preprocessing.dataset import load_dataset
from torch.utils.data import ConcatDataset

model = load_model(CPMPTransformer, CONFIG["model_name"])
print(f"Modelo '{CONFIG['model_name']}' cargado — hiperparámetros: {model.hyperparams}")

## 2. Generación de instancias base (opcional — el loop regenera por iteración)

> **Nota:** En el flujo actual, cada iteración genera sus propias instancias con seed distinto (`42 + iter`). Esta celda solo crea las instancias del seed base (42) si no existen — útil como respaldo / referencia. Puede saltarse.

In [3]:
from settings import INSTANCE_FOLDER

for (folder, S, H, N) in CONFIG["instance_sets"]:
    folder_path = INSTANCE_FOLDER / folder
    if folder_path.exists() and len(list(folder_path.iterdir())) >= CONFIG["instances_per_set"]:
        print(f"  {folder}: ya existe ({len(list(folder_path.iterdir()))} instancias) — saltando")
        continue
    generate_instances(folder, H, S, N, CONFIG["instances_per_set"], seed=42)
    print(f"  {folder}: {CONFIG['instances_per_set']} instancias generadas (S={S}, H={H}, N={N})")

  ExIt_E4: ya existe (5000 instancias) — saltando
  ExIt_E5: ya existe (5000 instancias) — saltando
  ExIt_E6: ya existe (5000 instancias) — saltando
  ExIt_E7: ya existe (5000 instancias) — saltando
  ExIt_5x5: ya existe (5000 instancias) — saltando
  ExIt_6x6: ya existe (5000 instancias) — saltando
  ExIt_7x7: ya existe (5000 instancias) — saltando
  ExIt_10x8: ya existe (5000 instancias) — saltando
  ExIt_10x7: ya existe (5000 instancias) — saltando
  ExIt_8x7: ya existe (5000 instancias) — saltando


## 3. Evaluación baseline en CVS

In [4]:
import math, time
from solvers.model import ModelSolver

def run_cvs_benchmark(model, max_steps=100):
    solver = ModelSolver(model)
    cvs_path = INSTANCE_FOLDER / "benchmarks" / "CVS"
    folders = sorted(d for d in os.listdir(cvs_path) if (cvs_path / d).is_dir())

    all_solved, all_steps = [], []
    for folder_name in folders:
        H_real, S_real = [int(x) for x in folder_name.split('-')]
        H_inf = H_real + 2
        dat_files = sorted(str(cvs_path / folder_name / f)
                           for f in os.listdir(cvs_path / folder_name)
                           if f.endswith('.dat'))
        for fp in dat_files:
            try:
                solved, steps = solver.solve_from_path(fp, H_inf, max_steps)
            except Exception:
                solved, steps = False, max_steps
            all_solved.append(solved)
            all_steps.append(steps)

    n = len(all_solved)
    n_solved = sum(all_solved)
    avg_steps = sum(s for s, ok in zip(all_steps, all_solved) if ok) / max(n_solved, 1)
    return {"solve_rate": n_solved / n, "mean_steps": avg_steps, "n": n, "n_solved": n_solved}

print("Evaluando baseline en CVS...")
baseline = run_cvs_benchmark(model, CONFIG["max_steps"])
print(f"Baseline  →  Solve rate: {baseline['solve_rate']:.1%}  |  Mean steps: {baseline['mean_steps']:.2f}  ({baseline['n_solved']}/{baseline['n']})")

# Historial de resultados por iteración
history = [{"iteration": 0, **baseline}]

Evaluando baseline en CVS...


/mnt/data/Proyectos/Universidad/CPMP-Transformer/.venv/lib/python3.14/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Baseline  →  Solve rate: 90.1%  |  Mean steps: 30.42  (757/840)


## 4. Loop de Expert Iteration

Cada iteración hace:
1. **Generar instancias frescas** con `seed = 42 + iter` (carpetas `{folder}_iter{i}`)
2. **Etiquetar con BSG (beam=5)** — el expert real, no el modelo
3. **Construir replay buffer** = BSG base (~250k) + datos ExIt-BSG de iter 1..i
4. **Fine-tune** del modelo actual sobre el dataset combinado
5. **Evaluar** en CVS + **guardar checkpoint** `v9_ExIt_iter{i}.pth`

In [ ]:
from training.metrics import Accuracy
from settings import DATA_FOLDER, INSTANCE_FOLDER

# ── Pre-cargar dataset BSG base (250k SL original, no cambia entre iteraciones) ──
print("Cargando dataset BSG base (replay buffer permanente)...")
bsg_base_datasets = []
for name in CONFIG["bsg_base_datasets"]:
    ds = load_dataset(f"{name}.data")
    bsg_base_datasets.append(ds)
    print(f"  {name} → {len(ds)} muestras")
bsg_base_total = sum(len(d) for d in bsg_base_datasets)
print(f"  Total BSG base: {bsg_base_total} muestras")

# ── Loop de Expert Iteration con BSG como expert real + replay buffer ───────
for iteration in range(1, CONFIG["iterations"] + 1):
    print(f"\n{'='*60}")
    print(f"  ITERACIÓN {iteration}/{CONFIG['iterations']}")
    print(f"{'='*60}")

    # ── 1. Generar instancias frescas por iteración (seed distinto) ────────
    print(f"\n[1/4] Generando instancias frescas (seed=42+{iteration})...")
    iter_folders = []
    for (folder, S, H, N) in CONFIG["instance_sets"]:
        iter_folder = f"{folder}_iter{iteration}"
        folder_path = INSTANCE_FOLDER / iter_folder
        if folder_path.exists() and len(list(folder_path.iterdir())) >= CONFIG["instances_per_set"]:
            print(f"  {iter_folder}: ya existe — saltando")
        else:
            generate_instances(iter_folder, H, S, N, CONFIG["instances_per_set"], seed=42 + iteration)
            print(f"  {iter_folder}: {CONFIG['instances_per_set']} instancias (S={S}, H={H}, N={N})")
        iter_folders.append((iter_folder, S, H, N))

    # ── 2. Etiquetar instancias con BSG (beam=5) — expert real ─────────────
    print(f"\n[2/4] Etiquetando con BSG (beam={CONFIG['bsg_beams']})...")
    for (folder, S, H, N) in iter_folders:
        output_name = f"{folder}_BSG"
        output_path = DATA_FOLDER / f"{output_name}.data"
        if output_path.exists():
            print(f"  {output_name}.data ya existe — saltando")
            continue
        generate_data(
            folder, H, CONFIG["max_steps"],
            EnrichedStackMatrix5DAdapter(),
            DefaultMovesAdapter(),
            output_name=output_name,
            beams=CONFIG["bsg_beams"],
        )

    # ── 3. Construir replay buffer: BSG base + datos ExIt-BSG acumulados ───
    print(f"\n[3/4] Construyendo replay buffer...")
    datasets = list(bsg_base_datasets)  # BSG base siempre presente
    for j in range(1, iteration + 1):
        for (folder, *_) in CONFIG["instance_sets"]:
            name = f"{folder}_iter{j}_BSG"
            ds_path = DATA_FOLDER / f"{name}.data"
            if ds_path.exists():
                ds = load_dataset(f"{name}.data")
                datasets.append(ds)
    combined = ConcatDataset(datasets)
    total_samples = len(combined)
    test_size = int(total_samples * CONFIG["test_ratio"])
    train_size = total_samples - test_size
    print(f"  Datasets: {len(datasets)}  |  Total: {total_samples}  |  train: {train_size}  |  test: {test_size}")

    # ── 4. Fine-tune ───────────────────────────────────────────────────────
    print(f"\n[4/4] Fine-tuning...")
    model = train(
        model,
        epochs       = CONFIG["epochs_per_iter"],
        dataset      = combined,
        train_size   = train_size,
        test_size    = test_size,
        batch_size   = CONFIG["batch_size"],
        learning_rate= CONFIG["learning_rate"],
        weight_decay = CONFIG["weight_decay"],
        patience     = CONFIG["patience"],
        metrics      = [Accuracy()],
        seed         = 42 + iteration,
    )

    # ── 5. Evaluar en CVS ─────────────────────────────────────────────────
    print(f"\nEvaluando en CVS...")
    result = run_cvs_benchmark(model, CONFIG["max_steps"])
    result["iteration"] = iteration
    history.append(result)

    prev = history[-2]
    delta_sr    = result["solve_rate"] - prev["solve_rate"]
    delta_steps = result["mean_steps"] - prev["mean_steps"]
    print(f"  Iter {iteration}  →  Solve rate: {result['solve_rate']:.1%}  "
          f"({delta_sr:+.1%})  |  Mean steps: {result['mean_steps']:.2f}  ({delta_steps:+.2f})")

    # ── 6. Guardar checkpoint ─────────────────────────────────────────────
    ckpt_name = f"{CONFIG['output_name']}_iter{iteration}"
    save_model(model, ckpt_name)
    print(f"  Checkpoint guardado: {ckpt_name}.pth")


## 5. Resultados por iteración

In [ ]:
print(f"\n{'Iter':>6}  {'Solve Rate':>12}  {'Mean Steps':>12}  {'Δ Solve Rate':>14}  {'Δ Steps':>10}")
print("─" * 62)
for i, r in enumerate(history):
    label = "baseline" if r["iteration"] == 0 else f"iter {r['iteration']:>2}"
    if i == 0:
        print(f"{label:>6}  {r['solve_rate']:>11.1%}  {r['mean_steps']:>12.2f}  {'—':>14}  {'—':>10}")
    else:
        prev = history[i - 1]
        d_sr    = r["solve_rate"]  - prev["solve_rate"]
        d_steps = r["mean_steps"]  - prev["mean_steps"]
        print(f"{label:>6}  {r['solve_rate']:>11.1%}  {r['mean_steps']:>12.2f}  "
              f"{d_sr:>+13.1%}  {d_steps:>+9.2f}")
print("─" * 62)

best = min(history, key=lambda r: r["mean_steps"])
print(f"\nMejor iteración: iter {best['iteration']}  →  "
      f"Solve rate: {best['solve_rate']:.1%}  |  Mean steps: {best['mean_steps']:.2f}")